In [4]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()
engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}?charset=utf8mb4"
)

OUT = Path("../data/processed")
OUT.mkdir(parents=True, exist_ok=True)

In [5]:
weather_raw = pd.read_sql("SELECT * FROM raw_weather", engine)
print("shape:", weather_raw.shape)

shape: (1472, 6)


In [6]:
weather_clean = weather_raw.copy()
weather_clean["date"] = pd.to_datetime(weather_clean["date"], errors="coerce")
for col in ["temp_max", "temp_min", "precipitation"]:
    weather_clean[col] = pd.to_numeric(weather_clean[col], errors="coerce")

print("missing before:")
print(weather_clean[["temp_max", "temp_min", "precipitation"]].isna().sum())

n0 = len(weather_clean)
weather_clean = weather_clean.drop_duplicates(subset=["city", "date"])
print("duplicates removed:", n0 - len(weather_clean))

bad = (weather_clean["temp_min"] > weather_clean["temp_max"]) | (weather_clean["precipitation"] < 0)
print("impossible rows:", int(bad.sum()))

# fill small gaps in temperature from neighbouring days (max 2 days), rain stays missing
weather_clean = weather_clean.sort_values(["city", "date"]).reset_index(drop=True)
for col in ["temp_max", "temp_min"]:
    weather_clean[col] = weather_clean.groupby("city")[col].transform(
        lambda s: s.interpolate(limit=2, limit_direction="both"))

weather_clean["temp_mean"] = (weather_clean["temp_max"] + weather_clean["temp_min"]) / 2
weather_clean["calendar_date"] = weather_clean["date"] + pd.DateOffset(years=1)
weather_clean = weather_clean.drop(columns=["note"], errors="ignore")

print("\nmissing after:")
print(weather_clean[["temp_max", "temp_min", "precipitation"]].isna().sum())
print(weather_clean.groupby("city")["calendar_date"].agg(["min", "max", "size"]))

missing before:
temp_max         0
temp_min         0
precipitation    0
dtype: int64
duplicates removed: 0
impossible rows: 0

missing after:
temp_max         0
temp_min         0
precipitation    0
dtype: int64
                 min        max  size
city                                 
Amsterdam 2026-09-17 2027-09-19   368
Barcelona 2026-09-17 2027-09-19   368
Berlin    2026-09-17 2027-09-19   368
Columbus  2026-09-17 2027-09-19   368


In [7]:
weather_clean.to_csv(OUT / "weather_clean.csv", index=False)
weather_clean.to_parquet(OUT / "weather_clean.parquet", index=False)